# Interpretation of Results and Explainable AI (Task 7)

## Purpose

Task 7 requires the decision-making process of the LCS to be examined rather
than only its predictive performance. This notebook analyses the rule
population produced by the improved eLCS system in notebook 08, translates
individual classifiers into readable statements, and assesses whether the
resulting explanations are trustworthy and clinically meaningful.

The distinguishing property of a Learning Classifier System is that its model
is a population of explicit IF-THEN rules rather than a set of weights or an
ensemble of trees. Each rule specifies conditions on a subset of features,
predicts a class, and carries its own accuracy and match statistics. A
prediction can therefore be traced to the specific rules that matched the
record, which is the property this notebook examines.

## Structure

1. Overview of the rule population
2. Translating classifiers into readable rules
3. Sample rules discussed in detail
4. Which features the population relies on
5. Whether the explanations can be trusted

## 1. Setup and rule population overview

In [ ]:
import re
import sys
from pathlib import Path

sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

results_dir = Path("../results")
rules = pd.read_csv(results_dir / "improved_elcs_rules.csv")

print(f"Rules in the final population: {len(rules)}")
print("\nPredicted class:")
print(rules["Diabetes_binary"].value_counts().sort_index())

print("\nRule statistics:")
print(
    rules[["Accuracy", "Numerosity", "Specificity", "Match Count"]]
    .describe()
    .round(3)
)

### What the columns mean

| Column | Meaning |
|---|---|
| Specified Attribute Names | The features this rule places a condition on |
| Specified Values | The condition on each of those features |
| Diabetes_binary | The class the rule predicts |
| Accuracy | Proportion of matched training records the rule classified correctly |
| Numerosity | Number of copies of this rule in the population, a measure of how strongly it was reinforced |
| Specificity | Proportion of the 21 features the rule constrains |
| Match Count | Training records the rule matched |

A rule with low specificity constrains few features and therefore matches many
records, giving a general statement. A rule with high specificity constrains
many features and describes a narrow subgroup.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(rules["Accuracy"], bins=30, color="#4c72b0", edgecolor="white")
axes[0].set_title("Rule accuracy")
axes[0].set_xlabel("Accuracy on matched training records")

axes[1].hist(
    rules["Specificity"] * 21, bins=range(0, 12),
    color="#55a868", edgecolor="white", align="left",
)
axes[1].set_title("Conditions per rule")
axes[1].set_xlabel("Number of features constrained")

axes[2].hist(
    np.log10(rules["Match Count"].clip(lower=1)),
    bins=30, color="#c44e52", edgecolor="white",
)
axes[2].set_title("Rule coverage")
axes[2].set_xlabel("log10(training records matched)")

for ax in axes:
    ax.set_ylabel("Number of rules")

plt.tight_layout()
plt.show()

## 2. Translating classifiers into readable rules

The exported population stores conditions in a compact form: a list of
attribute names and a matching list of values, where a discrete condition is a
single value and a continuous condition is an interval such as `[18.4, 31.2]`.

Two adjustments make the rules readable.

First, eLCS widens intervals during evolution and does not constrain them to
the range actually present in the data, so bounds such as `[-0.42, 2.82]` on
GenHlth appear even though GenHlth only takes values 1 to 5. An interval that
extends beyond the observed range is not a claim about impossible values; it
simply means the bound is not binding. Intervals are therefore clipped to the
observed range and, where a bound is no longer binding, expressed as a
one-sided condition.

Second, the coded values are replaced with their survey meanings, so that
`GenHlth <= 2` is shown as "general health is very good or excellent".

In [ ]:
# Observed range of every feature, used to decide whether a bound is binding
test_data = pd.read_csv(
    "../data/processed/elcs/diabetes_elcs_test_unchanged.csv"
)
features = test_data.drop(columns=["Diabetes_binary"])

feature_ranges = {
    column: (features[column].min(), features[column].max())
    for column in features.columns
}

# Plain-language meaning of the coded values
binary_meaning = {
    "HighBP": ("no high blood pressure", "high blood pressure"),
    "HighChol": ("no high cholesterol", "high cholesterol"),
    "CholCheck": ("no cholesterol check in 5 years", "cholesterol checked in 5 years"),
    "Smoker": ("not a smoker", "smoker"),
    "Stroke": ("no history of stroke", "history of stroke"),
    "HeartDiseaseorAttack": ("no heart disease", "heart disease or heart attack"),
    "PhysActivity": ("no recent physical activity", "physically active"),
    "Fruits": ("does not eat fruit daily", "eats fruit daily"),
    "Veggies": ("does not eat vegetables daily", "eats vegetables daily"),
    "HvyAlcoholConsump": ("not a heavy drinker", "heavy alcohol consumption"),
    "AnyHealthcare": ("no healthcare coverage", "has healthcare coverage"),
    "NoDocbcCost": ("could afford to see a doctor", "could not afford to see a doctor"),
    "DiffWalk": ("no difficulty walking", "difficulty walking or climbing stairs"),
    "Sex": ("female", "male"),
}

units = {
    "BMI": "BMI",
    "GenHlth": "general health rating (1 = excellent, 5 = poor)",
    "MentHlth": "days of poor mental health in the last 30",
    "PhysHlth": "days of poor physical health in the last 30",
    "Age": "age bracket (1 = 18-24, 13 = 80+)",
    "Education": "education level (1 = none, 6 = college graduate)",
    "Income": "income bracket (1 = under $10k, 8 = $75k+)",
}


def split_conditions(value_string):
    """Split the value list, keeping intervals such as [1.5,3.2] intact."""
    return [
        part.strip()
        for part in re.findall(r"\[[^\]]*\]|[^,\s][^,]*", str(value_string))
    ]


def describe_condition(attribute, condition):
    """Turn one attribute-condition pair into a readable phrase."""
    condition = condition.strip()

    if condition.startswith("["):
        low, high = [float(v) for v in condition.strip("[]").split(",")]
        observed_low, observed_high = feature_ranges[attribute]

        # A bound outside the observed range does not constrain anything
        low_binds = low > observed_low
        high_binds = high < observed_high

        label = units.get(attribute, attribute)

        if low_binds and high_binds:
            return f"{label} between {low:.1f} and {high:.1f}"
        if high_binds:
            return f"{label} at most {high:.1f}"
        if low_binds:
            return f"{label} at least {low:.1f}"
        return f"{label} unconstrained"

    if attribute in binary_meaning:
        return binary_meaning[attribute][int(float(condition))]

    return f"{attribute} = {condition}"


def readable_rule(row):
    """Render a full classifier as an IF-THEN statement."""
    attributes = [a.strip() for a in str(row["Specified Attribute Names"]).split(",")]
    conditions = split_conditions(row["Specified Values"])

    phrases = [
        describe_condition(attribute, condition)
        for attribute, condition in zip(attributes, conditions)
    ]
    phrases = [p for p in phrases if not p.endswith("unconstrained")]

    outcome = (
        "diabetes or prediabetes"
        if row["Diabetes_binary"] == 1
        else "no diabetes"
    )

    return "IF " + " AND ".join(phrases) + f" THEN {outcome}"


# Check the translation against the raw export
example = rules.sort_values("Match Count", ascending=False).iloc[0]
print("Raw attributes: ", example["Specified Attribute Names"])
print("Raw conditions: ", example["Specified Values"])
print()
print(readable_rule(example))

## 3. Sample rules

Task 7 requires at least three classifiers to be discussed. Rules are selected
on two criteria: accuracy on the training records they matched, and coverage,
since a rule that matches only a handful of records explains very little even
if it is perfectly accurate. Rules matching fewer than 200 training records are
therefore excluded from the selection.

Three groups are examined: the most reliable rules predicting no diabetes, the
most reliable rules predicting diabetes, and the least reliable rules in the
population, which are included because a rule population is not uniformly
trustworthy and the weak rules are as informative as the strong ones.

In [ ]:
covered = rules[rules["Match Count"] >= 200].copy()
covered["readable"] = covered.apply(readable_rule, axis=1)

print(f"Rules matching at least 200 training records: {len(covered)}")


def show_rules(subset, heading, n=3):
    print("\n" + "=" * 78)
    print(heading)
    print("=" * 78)

    for _, row in subset.head(n).iterrows():
        print(f"\n{row['readable']}")
        print(
            f"   accuracy {row['Accuracy']:.3f} | "
            f"matched {int(row['Match Count']):,} records | "
            f"numerosity {int(row['Numerosity'])} | "
            f"{len(str(row['Specified Attribute Names']).split(',')):d} conditions"
        )


show_rules(
    covered[covered["Diabetes_binary"] == 0].sort_values("Accuracy", ascending=False),
    "MOST ACCURATE RULES PREDICTING NO DIABETES",
)

show_rules(
    covered[covered["Diabetes_binary"] == 1].sort_values("Accuracy", ascending=False),
    "MOST ACCURATE RULES PREDICTING DIABETES OR PREDIABETES",
)

show_rules(
    covered.sort_values("Accuracy"),
    "LEAST ACCURATE RULES IN THE POPULATION",
)

### Discussion of the sample rules

*Complete this section after running the notebook, using the rules printed
above.*

For each of the three selected rules, address:

* What the rule says in clinical terms, and which features it relies on.
* Whether the direction of the relationship agrees with established diabetes
  risk factors, and with the exploratory analysis in notebook 03.
* How much of the population the rule covers, and how accurate it is on the
  records it matches.
* For the weak rules, why the system retains classifiers that are barely better
  than chance, and what that implies about reading any single rule in
  isolation.

## 4. Which features the population relies on

A rule population does not provide a single feature-importance score in the way
a tree ensemble does. An equivalent view is obtained by counting how often each
feature is constrained, weighting each occurrence by the numerosity of the rule
so that strongly reinforced classifiers count for more.

Because the improved system specifies attribute types explicitly, this also
shows whether the seven range-based features are being used differently from
the fourteen discrete ones.

In [ ]:
feature_usage = {column: 0.0 for column in features.columns}
feature_rules = {column: 0 for column in features.columns}

for _, row in rules.iterrows():
    attributes = [a.strip() for a in str(row["Specified Attribute Names"]).split(",")]
    for attribute in attributes:
        if attribute in feature_usage:
            feature_usage[attribute] += row["Numerosity"]
            feature_rules[attribute] += 1

usage = (
    pd.DataFrame({
        "weighted_usage": pd.Series(feature_usage),
        "rules_using_it": pd.Series(feature_rules),
    })
    .sort_values("weighted_usage", ascending=False)
)

usage["share_of_rules"] = (usage["rules_using_it"] / len(rules)).round(3)

display(usage)

In [ ]:
plt.figure(figsize=(10, 7))

range_based = ["BMI", "GenHlth", "MentHlth", "PhysHlth", "Age", "Education", "Income"]
colours = [
    "#c44e52" if name in range_based else "#4c72b0"
    for name in usage.index
]

plt.barh(range(len(usage)), usage["weighted_usage"], color=colours)
plt.yticks(range(len(usage)), usage.index)
plt.gca().invert_yaxis()
plt.xlabel("Total numerosity of rules constraining the feature")
plt.title("Feature use across the rule population (red = range-based attributes)")
plt.tight_layout()
plt.show()

## 5. Are the explanations trustworthy?

Interpretability is only useful if the explanations are reliable. Three checks
are applied.

**Coverage.** If large parts of the population are described only by rules with
very few matches, then most predictions rest on weak evidence.

**Accuracy spread.** If the population contains many rules barely better than
chance, a reader who happens to inspect one of them would be misled.

**Agreement with domain knowledge.** If the most reinforced rules contradict
established diabetes risk factors, the explanations are not trustworthy even
where the predictions are accurate.

In [ ]:
# Check 1: how concentrated is coverage
coverage_share = (
    rules.sort_values("Match Count", ascending=False)["Match Count"].cumsum()
    / rules["Match Count"].sum()
)

rules_for_half = int((coverage_share <= 0.5).sum()) + 1

print("Coverage concentration")
print(f"  Rules needed to account for half of all matches: {rules_for_half}")
print(f"  That is {rules_for_half / len(rules):.1%} of the population")
print(f"  Median matches per rule: {rules['Match Count'].median():,.0f}")

# Check 2: accuracy spread
weak = (rules["Accuracy"] < 0.6).sum()
strong = (rules["Accuracy"] > 0.8).sum()

print("\nAccuracy spread")
print(f"  Rules below 60% accuracy: {weak} ({weak / len(rules):.1%})")
print(f"  Rules above 80% accuracy: {strong} ({strong / len(rules):.1%})")
print(f"  Numerosity-weighted mean accuracy: "
      f"{np.average(rules['Accuracy'], weights=rules['Numerosity']):.3f}")

In [ ]:
# Check 3: do the strongest rules agree with known risk factors?
# Established risk factors should appear predominantly in rules predicting
# diabetes when the risk factor is present.
risk_factors = ["HighBP", "HighChol", "HeartDiseaseorAttack", "DiffWalk", "Stroke"]

checks = []

for factor in risk_factors:
    present_positive = 0
    present_negative = 0

    for _, row in rules.iterrows():
        attributes = [a.strip() for a in str(row["Specified Attribute Names"]).split(",")]
        if factor not in attributes:
            continue

        conditions = split_conditions(row["Specified Values"])
        condition = conditions[attributes.index(factor)]
        if condition.startswith("["):
            continue

        if int(float(condition)) == 1:  # risk factor present
            if row["Diabetes_binary"] == 1:
                present_positive += row["Numerosity"]
            else:
                present_negative += row["Numerosity"]

    total = present_positive + present_negative
    checks.append({
        "risk_factor": factor,
        "predicts_diabetes": present_positive,
        "predicts_no_diabetes": present_negative,
        "share_predicting_diabetes": present_positive / total if total else np.nan,
    })

agreement = pd.DataFrame(checks).round(3)

print("When the risk factor is present, what does the rule predict?\n")
print(agreement.to_string(index=False))

## 6. Findings

*Complete this section after running the notebook.*

Points to address:

* **How LCS rules support interpretability.** A prediction can be traced to the
  specific rules that matched the record, and each rule can be read as a
  statement about a subgroup. Contrast this with the conventional models: the
  Random Forest that outperformed the LCS offers feature importances but no
  comparable account of an individual decision.
* **The cost of that interpretability.** The population contains over 1,800
  rules, so the model as a whole is not simple, even though each rule is. State
  honestly whether a clinician could work with this, and what would make it
  more usable, for example rule compaction.
* **The three sample rules**, discussed in clinical terms, including whether
  they agree with the patterns found in the exploratory analysis.
* **Whether the explanations are trustworthy**, drawing on the three checks
  above. Note in particular the proportion of weak rules and what that implies
  about reading any single classifier in isolation.
* **Practical meaning.** The improved system reaches about 76% recall at about
  29% precision, so roughly two in three people it flags do not have diabetes.
  Discuss what an appropriate use of such a tool would be, and what it should
  not be used for.